In [ ]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from pathlib import Path
from datetime import datetime, timedelta

import os
import time
import json
import calendar
import requests
import pandas as pd

In [2]:
GDELT_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

class NewsScraper:

    def __init__(self, max_records_per_window=250, timeout=60, min_delay=15):
        self.max_records_per_window = max_records_per_window
        self.timeout = timeout
        self.min_delay = min_delay

        self.session = requests.Session()

        self.session.headers.update({
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "application/json, text/plain, */*",
        })

        self._last_request_time = 0


    def _throttle(self):
        #Ensure at least min_delay seconds between requests.
        elapsed = time.time() - self._last_request_time

        if elapsed < self.min_delay:
            time.sleep(self.min_delay - elapsed)


    def _monthly_windows(self, start, end):
        #Split the requested study period into monthly windows.
        start_dt = datetime.strptime(start, "%Y%m%d%H%M%S")
        end_dt = datetime.strptime(end, "%Y%m%d%H%M%S")

        windows = []

        cur = start_dt.replace(day=1,hour=0,minute=0,second=0)

        while cur <= end_dt:

            last_day = calendar.monthrange(cur.year,cur.month)[1]
            month_end = cur.replace(day=last_day,hour=23,minute=59,second=59)
            win_start = max(cur, start_dt)
            win_end = min(month_end, end_dt)

            windows.append((win_start, win_end))

            if cur.month == 12:
                cur = cur.replace(year=cur.year + 1,month=1)
            else:
                cur = cur.replace(month=cur.month + 1)

        return windows


    def _fetch_window(self, query, win_start, win_end, max_retries=7):
        """
        Request one GDELT time window.

        Returns
        -------
        list
            Successful request. May be empty if no articles matched.

        None
            Request failed after retries.
        """

        params = {
            "query": query,
            "mode": "ArtList",
            "maxrecords": self.max_records_per_window,
            "format": "json",
            "startdatetime": win_start.strftime("%Y%m%d%H%M%S"),
            "enddatetime": win_end.strftime("%Y%m%d%H%M%S"),
            "sort": "DateAsc",
        }

        backoff = self.min_delay

        for attempt in range(1, max_retries + 1):
            self._throttle()
            try:
                response = self.session.get(
                    GDELT_URL,
                    params=params,
                    timeout=self.timeout
                )

                self._last_request_time = time.time()

            except requests.RequestException as e:

                self._last_request_time = time.time()

                print(f"    Request error "
                    f"(attempt {attempt}/{max_retries}): {e}")

                if attempt < max_retries:
                    print(f"    Retrying in {backoff}s...")

                    time.sleep(backoff)
                    backoff *= 2

                continue

            # Successful HTTP response
            if response.status_code == 200:
                try:
                    data = response.json()
                except ValueError:
                    print("Could not parse JSON response.")
                    return None

                return data.get("articles", [])

            # GDELT rate limit
            if response.status_code == 429:
                print(f"429 rate-limited "
                    f"(attempt {attempt}/{max_retries})")

                if attempt < max_retries:
                    print(f"Backing off {backoff}s...")

                    time.sleep(backoff)
                    backoff *= 2

                continue


            # Other HTTP error
            print(
                f"    Unexpected status "
                f"{response.status_code}: "
                f"{response.text[:200]}"
            )

            return None

        print("Giving up after maximum retries.")

        return None


    @staticmethod
    def save(df, path):
        df.to_csv(path, index=False)

    @staticmethod
    def load(path):
        df = pd.read_csv(path)

        if "published_at" in df.columns:
            df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce", utc=True)

        return df

In [ ]:
# ---------------------------------------------------------
# GME Historical News Collection
# ---------------------------------------------------------

NEWS_DIR = Path("../data/raw/news")
NEWS_DIR.mkdir(parents=True, exist_ok=True)

TICKER = "GME"

QUERY = ("GameStop GME stock Reddit WallStreetBets")

START_DATE = "20201201000000"
END_DATE = "20210331235959"

# ---------------------------------------------------------
# File paths
# ---------------------------------------------------------

CHECKPOINT_PATH = (NEWS_DIR/"news_gdelt_GME_checkpoint.csv")
STATE_PATH = (NEWS_DIR/"news_gdelt_GME_state.json")
FINAL_PATH = (NEWS_DIR/"news_gdelt_GME.csv")

# ---------------------------------------------------------
# Initialize scraper
# ---------------------------------------------------------

gdelt = NewsScraper()

# ---------------------------------------------------------
# Load article checkpoint
# ---------------------------------------------------------

if CHECKPOINT_PATH.exists():

    checkpoint_df = NewsScraper.load(CHECKPOINT_PATH)

    print(f"Resuming from checkpoint: {len(checkpoint_df)} articles already collected.")

else:
    checkpoint_df = pd.DataFrame()

    print("No checkpoint found. Starting new collection.")

# ---------------------------------------------------------
# Load completed-window state
# ---------------------------------------------------------

if STATE_PATH.exists():

    with open(STATE_PATH, "r") as file:
        completed_windows = set(json.load(file))

else:
    completed_windows = set()

# ---------------------------------------------------------
# Helper to build a window's dictionary key
# ---------------------------------------------------------

def make_window_key(win_start, win_end):
    return (f"{win_start.strftime('%Y%m%d%H%M%S')}_{win_end.strftime('%Y%m%d%H%M%S')}")

# ---------------------------------------------------------
# Build the work queue from monthly windows
# (skip any leaf window already completed in a previous run)
# ---------------------------------------------------------

month_windows = gdelt._monthly_windows(START_DATE, END_DATE)

queue = [w for w in month_windows if make_window_key(*w) not in completed_windows]

failed_windows = []

print(f"\n{len(queue)} top-level window(s) to process "
      f"({len(month_windows) - len(queue)} already fully completed)\n")

# ---------------------------------------------------------
# Process the queue.
#
#   - success, under the 250-cap  -> save immediately, mark done
#   - success, at the 250-cap     -> split in half, push both
#                                     halves to the FRONT of the
#                                     queue so this month finishes
#                                     before moving to the next one
#   - failure after all retries   -> record as failed and move on
#                                     to the NEXT window in the queue,
#                                     instead of stopping everything
# ---------------------------------------------------------

while queue:

    win_start, win_end = queue.pop(0)
    key = make_window_key(win_start, win_end)

    if key in completed_windows:
        print(f"\nSkipping (already completed): {win_start} → {win_end}")
        continue

    print(f"\nWindow {win_start} → {win_end}")

    articles = gdelt._fetch_window(QUERY, win_start, win_end)

    # -----------------------------------------------------
    # API failure — don't stop, just leave this window for
    # a later run and move on to the rest of the queue
    # -----------------------------------------------------

    if articles is None:

        print("  Failed after retries — leaving for a later run "
              "(nothing else is lost).")

        failed_windows.append((win_start, win_end))

        continue

    print(f"  Got {len(articles)} articles")

    # -----------------------------------------------------
    # Below the 250-record cap → this window is a finished
    # leaf. Save it and mark it complete right away.
    # -----------------------------------------------------

    if len(articles) < gdelt.max_records_per_window:

        if len(articles) > 0:

            batch_df = pd.DataFrame(articles)

            if "seendate" in batch_df.columns:
                batch_df["published_at"] = pd.to_datetime(
                    batch_df["seendate"], format="%Y%m%dT%H%M%SZ",
                    errors="coerce", utc=True
                )
            else:
                batch_df["published_at"] = pd.NaT

            if checkpoint_df.empty:
                checkpoint_df = batch_df.copy()
            else:
                checkpoint_df = pd.concat([checkpoint_df, batch_df], ignore_index=True)

            if "url" in checkpoint_df.columns:
                checkpoint_df = (
                    checkpoint_df.drop_duplicates(subset="url")
                    .sort_values("published_at")
                    .reset_index(drop=True)
                )

            checkpoint_df.to_csv(CHECKPOINT_PATH, index=False)

        completed_windows.add(key)

        with open(STATE_PATH, "w") as file:
            json.dump(sorted(completed_windows), file, indent=2)

        print(f"  Checkpoint saved | Total unique articles: {len(checkpoint_df)}")

    # -----------------------------------------------------
    # Hit the 250-record cap → split the window and push
    # both halves back onto the queue instead of this one
    # -----------------------------------------------------

    else:

        window_seconds = int((win_end - win_start).total_seconds())

        if window_seconds <= 86400:

            print("  WARNING: 250-record cap reached even at the "
                  "minimum window size — saving as-is.")

            batch_df = pd.DataFrame(articles)

            if "seendate" in batch_df.columns:
                batch_df["published_at"] = pd.to_datetime(
                    batch_df["seendate"], format="%Y%m%dT%H%M%SZ",
                    errors="coerce", utc=True
                )
            else:
                batch_df["published_at"] = pd.NaT

            if checkpoint_df.empty:
                checkpoint_df = batch_df.copy()
            else:
                checkpoint_df = pd.concat([checkpoint_df, batch_df], ignore_index=True)

            if "url" in checkpoint_df.columns:
                checkpoint_df = (
                    checkpoint_df.drop_duplicates(subset="url")
                    .sort_values("published_at")
                    .reset_index(drop=True)
                )

            checkpoint_df.to_csv(CHECKPOINT_PATH, index=False)

            completed_windows.add(key)

            with open(STATE_PATH, "w") as file:
                json.dump(sorted(completed_windows), file, indent=2)

            print(f"  Checkpoint saved | Total unique articles: {len(checkpoint_df)}")

        else:

            half_seconds = window_seconds // 2
            left_end = win_start + timedelta(seconds=half_seconds)
            right_start = left_end + timedelta(seconds=1)

            print(f"  250-record cap reached — splitting window.")
            print(f"      Left:  {win_start} → {left_end}")
            print(f"      Right: {right_start} → {win_end}")

            queue.insert(0, (right_start, win_end))
            queue.insert(0, (win_start, left_end))

# ---------------------------------------------------------
# Report progress and save the final dataset only when
# every window has been completed
# ---------------------------------------------------------

if failed_windows:

    print(f"\n{len(failed_windows)} window(s) still incomplete after this run:")

    for ws, we in failed_windows:
        print(f"  {ws} → {we}")

    print("\nRe-run this same cell later — completed windows are "
          "checkpointed and will be skipped automatically.")

else:

    print("\nAll windows have been completed.")

    if not checkpoint_df.empty:

        final_df = checkpoint_df.copy()

        final_df["ticker"] = TICKER

        if "url" in final_df.columns:
            final_df = (
                final_df.drop_duplicates(subset="url")
                .sort_values("published_at")
                .reset_index(drop=True)
            )

        final_df.to_csv(FINAL_PATH, index=False)

        print(f"\nFinal dataset saved: {len(final_df)} unique articles")
        print(f"Date range: {final_df['published_at'].min()} → {final_df['published_at'].max()}")

        # -------------------------------------------------
        # Quick validation
        # -------------------------------------------------

        print("\nDataset shape:")
        print(final_df.shape)
        print("\nDuplicate URLs:")
        print(final_df["url"].duplicated().sum())
        print("\nColumns:")
        print(final_df.columns.tolist())
        print("\nArticles by month:")
        print(final_df.set_index("published_at").resample("MS").size())

        display(final_df.head())

    else:
        print("\nCollection completed, but no matching articles were found.")

Resuming from checkpoint: 2073 articles already collected.

3 top-level window(s) to process (1 already fully completed)


Window 2021-01-01 00:00:00 → 2021-01-31 23:59:59
429 rate-limited (attempt 1/7)
Backing off 15s...
  Got 250 articles
  250-record cap reached — splitting window.
      Left:  2021-01-01 00:00:00 → 2021-01-16 11:59:59
      Right: 2021-01-16 12:00:00 → 2021-01-31 23:59:59

Skipping (already completed): 2021-01-01 00:00:00 → 2021-01-16 11:59:59

Window 2021-01-16 12:00:00 → 2021-01-31 23:59:59
429 rate-limited (attempt 1/7)
Backing off 15s...
429 rate-limited (attempt 2/7)
Backing off 30s...
429 rate-limited (attempt 3/7)
Backing off 60s...
429 rate-limited (attempt 4/7)
Backing off 120s...
429 rate-limited (attempt 5/7)
Backing off 240s...
  Got 250 articles
  250-record cap reached — splitting window.
      Left:  2021-01-16 12:00:00 → 2021-01-24 05:59:59
      Right: 2021-01-24 06:00:00 → 2021-01-31 23:59:59

Skipping (already completed): 2021-01-16 12:00:00 → 20

In [7]:
# ---------------------------------------------------------
# Check temporal coverage of collected GME news
# ---------------------------------------------------------

coverage_df = checkpoint_df.copy()

coverage_df["published_at"] = pd.to_datetime(
    coverage_df["published_at"],
    errors="coerce",
    utc=True
)

# Create date column
coverage_df["date"] = coverage_df["published_at"].dt.date


# Overall coverage
print("Total articles:", len(coverage_df))

print(
    "Date range:",
    coverage_df["published_at"].min(),
    "→",
    coverage_df["published_at"].max()
)


# ---------------------------------------------------------
# Articles by month
# ---------------------------------------------------------

print("\nArticles by month:")

monthly_counts = (
    coverage_df
    .set_index("published_at")
    .resample("MS")
    .size()
)

print(monthly_counts)


# ---------------------------------------------------------
# Articles by day
# ---------------------------------------------------------

daily_counts = (
    coverage_df
    .set_index("published_at")
    .resample("D")
    .size()
)

print("\nDaily coverage:")
display(
    daily_counts.to_frame("articles")
)


# ---------------------------------------------------------
# Days with and without articles
# ---------------------------------------------------------

study_days = pd.date_range(
    start="2020-12-01",
    end="2021-03-31",
    freq="D",
    tz="UTC"
)

days_with_articles = pd.DatetimeIndex(
    coverage_df["published_at"]
    .dt.normalize()
    .dropna()
    .unique()
)

days_without_articles = (
    study_days.difference(days_with_articles)
)

print("\nNumber of calendar days:", len(study_days))
print("Days with articles:", len(days_with_articles))
print("Days without articles:", len(days_without_articles))

print("\nDates without articles:")
print(days_without_articles)

Total articles: 2073
Date range: 2021-01-15 23:30:00+00:00 → 2021-04-01 20:30:00+00:00

Articles by month:
published_at
2021-01-01 00:00:00+00:00    965
2021-02-01 00:00:00+00:00    850
2021-03-01 00:00:00+00:00    254
2021-04-01 00:00:00+00:00      4
Freq: MS, dtype: int64

Daily coverage:


,articles
published_at,
2021-01-15 00:00:00+00:00,2
2021-01-16 00:00:00+00:00,0
2021-01-17 00:00:00+00:00,0
2021-01-18 00:00:00+00:00,0
2021-01-19 00:00:00+00:00,1
...,...
2021-03-28 00:00:00+00:00,3
2021-03-29 00:00:00+00:00,3
2021-03-30 00:00:00+00:00,6



Number of calendar days: 121
Days with articles: 73
Days without articles: 49

Dates without articles:
DatetimeIndex(['2020-12-01 00:00:00+00:00', '2020-12-02 00:00:00+00:00',
               '2020-12-03 00:00:00+00:00', '2020-12-04 00:00:00+00:00',
               '2020-12-05 00:00:00+00:00', '2020-12-06 00:00:00+00:00',
               '2020-12-07 00:00:00+00:00', '2020-12-08 00:00:00+00:00',
               '2020-12-09 00:00:00+00:00', '2020-12-10 00:00:00+00:00',
               '2020-12-11 00:00:00+00:00', '2020-12-12 00:00:00+00:00',
               '2020-12-13 00:00:00+00:00', '2020-12-14 00:00:00+00:00',
               '2020-12-15 00:00:00+00:00', '2020-12-16 00:00:00+00:00',
               '2020-12-17 00:00:00+00:00', '2020-12-18 00:00:00+00:00',
               '2020-12-19 00:00:00+00:00', '2020-12-20 00:00:00+00:00',
               '2020-12-21 00:00:00+00:00', '2020-12-22 00:00:00+00:00',
               '2020-12-23 00:00:00+00:00', '2020-12-24 00:00:00+00:00',
               '2020

In [11]:
news_raw = checkpoint_df.copy()

news_raw["published_at"] = pd.to_datetime(
    news_raw["published_at"],
    errors="coerce",
    utc=True
)

# Keep only records within the study period
news_raw = news_raw[
    (news_raw["published_at"] >= "2020-12-01") &
    (news_raw["published_at"] < "2021-04-01")
].copy()

# Add ticker
news_raw["ticker"] = "GME"

# Final validation
print("Shape:", news_raw.shape)

print(
    "Date range:",
    news_raw["published_at"].min(),
    "→",
    news_raw["published_at"].max()
)

print("\nDuplicate URLs:")
print(news_raw["url"].duplicated().sum())

print("\nArticles by month:")
print(
    news_raw
    .set_index("published_at")
    .resample("MS")
    .size()
)

Shape: (2069, 10)
Date range: 2021-01-15 23:30:00+00:00 → 2021-03-31 23:30:00+00:00

Duplicate URLs:
0

Articles by month:
published_at
2021-01-01 00:00:00+00:00    965
2021-02-01 00:00:00+00:00    850
2021-03-01 00:00:00+00:00    254
Freq: MS, dtype: int64


In [12]:
print("\nMissing values:")
print(
    news_raw[
        [
            "title",
            "url",
            "seendate",
            "domain",
            "language",
            "sourcecountry",
            "published_at"
        ]
    ].isna().sum()
)


Missing values:
title             0
url               0
seendate          0
domain            0
language          0
sourcecountry    19
published_at      0
dtype: int64


In [13]:
FINAL_PATH = NEWS_DIR / "news_gdelt_GME.csv"

news_raw.to_csv(
    FINAL_PATH,
    index=False
)

print(f"Saved {len(news_raw)} articles to:")
print(FINAL_PATH)

Saved 2069 articles to:
/Users/nlimsupt/Desktop/SUMMER/bia660_project/data/raw/news/news_gdelt_GME.csv
